# ASL Hand Sign MLP Classifier — Optimized for Raspberry Pi 5

**Project:** Train a tiny MLP on MediaPipe hand landmarks (63 features) to classify 28 ASL signs (A–Z + SPACE + DELETE).

**Target:** Raspberry Pi 5 (ARM Cortex-A76, Debian Bookworm, Python 3.11) — offline, privacy-first edge inference.

**Key design choices:**
- **Extremely small model** (1–2 hidden layers, <50 KB quantized) to fit in CPU cache
- **INT8 post-training quantization** for fast ARM64 CPU inference and minimal RAM
- **ReLU activations** (fastest on ARM — no exp/div operations)
- **Strong regularization** (Dropout + L2) to prevent overfitting on a small model
- **Minimal RPi 5 dependencies:** only `tflite-runtime` + `numpy` needed at inference time

In [ ]:
# ================================================================
# Install required packages (most pre-installed on Colab)
# ================================================================
!pip install -q tensorflow scikit-learn pandas numpy matplotlib seaborn

In [ ]:
# ================================================================
# Imports
# ================================================================
import os
import sys
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

print(f"Python:     {sys.version}")
print(f"TensorFlow: {tf.__version__}")
print(f"NumPy:      {np.__version__}")

# ================================================================
# Mount Google Drive (paths match landmarks extraction notebook)
# ================================================================
from google.colab import drive
drive.mount('/content/drive')

# ================================================================
# Paths — consistent with landmark extraction notebook
# CSV was saved to EDGE/ folder by the extraction pipeline
# ================================================================
DATA_DIR   = "/content/drive/MyDrive/Colab_Notebooks/EDGE"
CSV_PATH   = os.path.join(DATA_DIR, "combined_asl_landmarks.csv")
OUTPUT_DIR = os.path.join(DATA_DIR, "mlp_model")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ================================================================
# Load CSV
# 63 feature columns (x0,y0,z0 ... x20,y20,z20) + 1 label column
# ================================================================
df = pd.read_csv(CSV_PATH)
print(f"\nDataset shape: {df.shape}")
print(f"Feature columns: {list(df.columns[:6])} ... {list(df.columns[-4:])}")
print(f"\nClass distribution ({df['label'].nunique()} classes):")
print(df['label'].value_counts().sort_index())
df.head()

In [ ]:
# ================================================================
# DATA PREPROCESSING
# ================================================================

# --- Separate features (63 landmark coords) and labels ---
feature_cols = [c for c in df.columns if c != 'label']
X = df[feature_cols].values.astype(np.float32)   # (N, 63)
y_raw = df['label'].values                        # string labels

print(f"Features: {X.shape}, dtype: {X.dtype}")
print(f"Unique labels ({len(set(y_raw))}): {sorted(set(y_raw))}")

# --- Label Encoding: string → integer → one-hot ---
# LabelEncoder sorts alphabetically: A,B,...,Z,del,space
# We save the mapping so RPi 5 can decode predictions back to labels.
le = LabelEncoder()
y_int = le.fit_transform(y_raw)                    # (N,) integers
num_classes = len(le.classes_)
y_onehot = keras.utils.to_categorical(y_int, num_classes)  # (N, 28)

label_map = {int(i): str(c) for i, c in enumerate(le.classes_)}
label_map_path = os.path.join(OUTPUT_DIR, "label_map.json")
with open(label_map_path, 'w') as f:
    json.dump(label_map, f, indent=2)
print(f"\nLabel map saved: {label_map_path}")
print(f"Classes ({num_classes}): {list(label_map.values())}")

# --- Feature Normalization: StandardScaler ---
# Why: MediaPipe x/y are in [0,1] but z has a different, variable scale.
# StandardScaler (zero-mean, unit-variance) makes all 63 features comparable,
# which helps the small MLP converge faster and — critically — improves INT8
# quantization accuracy by giving the converter a consistent input range.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Save scaler params for RPi 5 inference (just numpy arrays — no joblib needed)
np.save(os.path.join(OUTPUT_DIR, "scaler_mean.npy"),  scaler.mean_.astype(np.float32))
np.save(os.path.join(OUTPUT_DIR, "scaler_scale.npy"), scaler.scale_.astype(np.float32))
print("Scaler params saved (mean + scale).")

# --- Stratified Train / Validation / Test Split: 80 / 10 / 10 ---
# Stratified ensures every class is proportionally represented in each split.
# This is important because some letters (e.g., uncommon signs) may have fewer samples.
X_train, X_temp, y_train, y_temp = train_test_split(
    X_scaled, y_onehot, test_size=0.2, random_state=42, stratify=y_int
)
# Split the 20% temp into 10% val + 10% test
y_temp_int = np.argmax(y_temp, axis=1)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp_int
)

print(f"\nSplit sizes — Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")

In [ ]:
# ================================================================
# BUILD MLP MODEL — Optimized for RPi 5 Edge Inference
# ================================================================
#
# Architecture rationale for ARM Cortex-A76 (RPi 5):
#   - 1–2 hidden layers: fewer MatMul ops → lower latency per inference
#   - ReLU activation: single comparison op on ARM (max(0,x)), fastest of all
#     activations. Swish/GELU use exp() which is ~10× slower per element.
#   - L2 regularization: keeps weights in a tight range → better INT8
#     quantization because the quantizer can use a narrower scale.
#   - Dropout: prevents overfitting on small model. Zero cost at inference
#     time (TFLite removes dropout nodes entirely from the graph).
#
# We test TWO configurations and pick the best accuracy/size trade-off:
#   Small:  64 → 32  (~4K params, ~4–6 KB INT8)   ← ultra-low latency
#   Medium: 128 → 64 (~12K params, ~10–14 KB INT8) ← better accuracy
# ================================================================

def build_mlp(input_dim, num_classes, hidden_sizes,
              dropout_rate=0.3, l2_weight=0.002, name="mlp"):
    """
    Build a small MLP for edge deployment.

    Args:
        input_dim:    Number of input features (63 for hand landmarks)
        num_classes:  Number of output classes (28 for ASL)
        hidden_sizes: List of hidden layer sizes, e.g. [64, 32]
        dropout_rate: Dropout probability (0.2–0.4 for small models)
        l2_weight:    L2 regularization strength
        name:         Model name for identification

    Returns:
        Keras Sequential model (uncompiled)
    """
    model = keras.Sequential(name=name)
    model.add(layers.Input(shape=(input_dim,)))

    for i, units in enumerate(hidden_sizes):
        model.add(layers.Dense(
            units,
            activation='relu',                              # Fastest on ARM64
            kernel_regularizer=regularizers.l2(l2_weight),  # Bounded weights → better INT8
            name=f"dense_{i}"
        ))
        model.add(layers.Dropout(dropout_rate, name=f"dropout_{i}"))

    # Output: softmax for multi-class probability distribution
    model.add(layers.Dense(num_classes, activation='softmax', name="output"))
    return model


# ================================================================
# Build both configurations for comparison
# ================================================================
configs = {
    "small_64_32":   [64, 32],    # Ultra-small for minimum latency
    "medium_128_64": [128, 64],   # Slightly larger for better accuracy
}

models = {}
for config_name, hidden_sizes in configs.items():
    model = build_mlp(
        input_dim=X_train.shape[1],
        num_classes=num_classes,
        hidden_sizes=hidden_sizes,
        dropout_rate=0.3,    # Moderate: balances regularization vs. capacity
        l2_weight=0.002,     # Keeps weights small without killing learning
        name=config_name
    )
    models[config_name] = model

    total_params = model.count_params()
    est_int8_kb = total_params / 1024  # ~1 byte per param in INT8

    print(f"\n{'='*60}")
    print(f"Config: {config_name} | Hidden: {hidden_sizes}")
    print(f"{'='*60}")
    model.summary()
    print(f"Estimated float32 size: {total_params * 4 / 1024:.1f} KB")
    print(f"Estimated INT8 size:    {est_int8_kb:.1f} KB")

In [ ]:
# ================================================================
# COMPILE & TRAIN BOTH MODELS
# ================================================================
#
# Training strategy for edge models:
#   - Adam optimizer, LR=1e-3: fast convergence for small MLPs.
#   - ReduceLROnPlateau: auto-lowers LR when val_loss stalls — squeezes
#     extra accuracy from tiny models without manual LR scheduling.
#   - EarlyStopping: prevents wasted epochs and overfitting.
#     Restores best weights automatically.
#   - Batch size 32: small batches give natural regularization noise.
#   - Top-3 accuracy: useful for UX — if top-1 is wrong, the correct
#     class is often in the top 3, allowing a "did you mean...?" prompt.
# ================================================================

BATCH_SIZE  = 32    # Small batch → implicit regularization, matches edge constraints
EPOCHS      = 80    # Max epochs; EarlyStopping will cut it short
INITIAL_LR  = 1e-3  # Standard starting LR for Adam on small models

histories = {}

for config_name, model in models.items():
    print(f"\n{'='*60}")
    print(f"Training: {config_name}")
    print(f"{'='*60}")

    # Compile with fresh metrics for each model
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=INITIAL_LR),
        loss='categorical_crossentropy',
        metrics=[
            'accuracy',
            keras.metrics.TopKCategoricalAccuracy(k=3, name='top_3_accuracy')
        ]
    )

    # Callbacks
    cb_list = [
        callbacks.EarlyStopping(
            monitor='val_loss',
            patience=10,                # Wait 10 epochs before giving up
            restore_best_weights=True,  # Roll back to best epoch automatically
            verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,     # Halve LR when plateau detected
            patience=5,     # Wait 5 epochs before reducing
            min_lr=1e-6,    # Don't go below this
            verbose=1
        ),
        callbacks.ModelCheckpoint(
            filepath=os.path.join(OUTPUT_DIR, f"{config_name}_best.keras"),
            monitor='val_accuracy',
            save_best_only=True,
            verbose=0
        ),
    ]

    # Train
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=cb_list,
        verbose=1
    )
    histories[config_name] = history

# ================================================================
# Plot training curves for both configurations
# ================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for config_name, history in histories.items():
    axes[0].plot(history.history['accuracy'],     label=f'{config_name} train')
    axes[0].plot(history.history['val_accuracy'],  '--', label=f'{config_name} val')
    axes[1].plot(history.history['loss'],           label=f'{config_name} train')
    axes[1].plot(history.history['val_loss'],       '--', label=f'{config_name} val')

axes[0].set_title('Accuracy');  axes[0].legend(); axes[0].set_xlabel('Epoch')
axes[1].set_title('Loss');      axes[1].legend(); axes[1].set_xlabel('Epoch')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=100)
plt.show()

In [ ]:
# ================================================================
# EVALUATE BOTH MODELS & SELECT THE BEST
# ================================================================
#
# Selection rule: prefer the smaller model (64→32) unless the medium
# model (128→64) is more than 1% better on test accuracy.
# On RPi 5, a few KB smaller = better CPU cache utilization.
# ================================================================

results = {}
for config_name, model in models.items():
    loss, acc, top3 = model.evaluate(X_test, y_test, verbose=0)
    results[config_name] = {'loss': loss, 'accuracy': acc, 'top3_accuracy': top3}
    print(f"{config_name}: Test Accuracy={acc:.4f}, Top-3={top3:.4f}, Loss={loss:.4f}")

# Select best model — prefer smaller if accuracy difference < 1%
small_acc  = results["small_64_32"]["accuracy"]
medium_acc = results["medium_128_64"]["accuracy"]

if medium_acc - small_acc > 0.01:
    best_name = "medium_128_64"
    print(f"\n→ Selected MEDIUM (128→64): +{(medium_acc - small_acc)*100:.1f}% accuracy justifies extra size.")
else:
    best_name = "small_64_32"
    print(f"\n→ Selected SMALL (64→32): <1% difference — prefer smaller for RPi 5 cache efficiency.")

best_model = models[best_name]

# ================================================================
# Detailed classification report
# ================================================================
y_pred_probs = best_model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)
class_names = [label_map[i] for i in range(num_classes)]

print(f"\nClassification Report ({best_name}):")
print(classification_report(y_true, y_pred, target_names=class_names))

# ================================================================
# Confusion Matrix — identify which signs are confused
# ================================================================
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix — {best_name}')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"), dpi=100)
plt.show()

In [ ]:
# ================================================================
# SAVE KERAS MODEL + CONVERT TO TFLITE
# ================================================================

# --- Save full Keras model (.keras format, recommended for TF 2.x) ---
keras_path = os.path.join(OUTPUT_DIR, "asl_mlp.keras")
best_model.save(keras_path)
print(f"Keras model saved: {keras_path}")

# --- Also save as .h5 for legacy compatibility ---
h5_path = os.path.join(OUTPUT_DIR, "asl_mlp.h5")
best_model.save(h5_path)
print(f"H5 model saved:    {h5_path}")

# ================================================================
# TFLITE FLOAT32 — Baseline (fallback if INT8 has issues)
# ================================================================
converter_f32 = tf.lite.TFLiteConverter.from_keras_model(best_model)
tflite_f32 = converter_f32.convert()
f32_path = os.path.join(OUTPUT_DIR, "asl_mlp_float32.tflite")
with open(f32_path, 'wb') as f:
    f.write(tflite_f32)
print(f"\nFloat32 TFLite saved: {f32_path}")

# ================================================================
# TFLITE INT8 — Full Integer Quantization (Primary for RPi 5)
# ================================================================
#
# Why INT8 for RPi 5:
#   - 4× smaller model: float32 weights → int8 = 75% size reduction
#   - 2–4× faster inference: ARM Cortex-A76 has efficient integer ALUs;
#     INT8 MatMul is significantly faster than float32 on CPU
#   - Lower memory bandwidth: smaller tensors → less DRAM traffic →
#     less thermal throttling under sustained inference
#
# Representative dataset: calibrates the quantization ranges by observing
# real activation values. This preserves accuracy much better than naive
# min/max quantization. 300 samples is enough for stable statistics.
# ================================================================

def representative_dataset_gen():
    """
    Yield representative samples for INT8 quantization calibration.
    Uses 300 random training samples — enough for stable activation
    statistics without making the conversion process slow.
    """
    np.random.seed(42)
    indices = np.random.choice(len(X_train), size=min(300, len(X_train)), replace=False)
    for i in indices:
        # Each sample must be float32 with shape (1, 63)
        yield [X_train[i:i+1].astype(np.float32)]

converter_int8 = tf.lite.TFLiteConverter.from_keras_model(best_model)

# Enable default optimizations (weight + activation quantization with rep dataset)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = representative_dataset_gen

# Keep float32 I/O for simplicity on RPi 5 — internal ops run as INT8.
# This means no manual quantize/dequantize of inputs needed in Python.
# The speed benefit is almost identical to full-integer I/O for this model size.

tflite_int8 = converter_int8.convert()
int8_path = os.path.join(OUTPUT_DIR, "asl_mlp_int8.tflite")
with open(int8_path, 'wb') as f:
    f.write(tflite_int8)
print(f"INT8 TFLite saved:   {int8_path}")

# ================================================================
# VERIFY TFLITE INT8 ACCURACY — ensure quantization didn't break it
# ================================================================
interpreter = tf.lite.Interpreter(model_path=int8_path)
interpreter.allocate_tensors()
input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

correct = 0
for i in range(len(X_test)):
    interpreter.set_tensor(input_details[0]['index'],
                           X_test[i:i+1].astype(np.float32))
    interpreter.invoke()
    pred = np.argmax(interpreter.get_tensor(output_details[0]['index']))
    if pred == y_true[i]:
        correct += 1

tflite_acc = correct / len(X_test)
keras_acc  = results[best_name]['accuracy']

print(f"\n{'='*50}")
print(f"QUANTIZATION ACCURACY CHECK")
print(f"{'='*50}")
print(f"Keras (float32):  {keras_acc:.4f}")
print(f"TFLite (INT8):    {tflite_acc:.4f}")
print(f"Accuracy drop:    {(keras_acc - tflite_acc)*100:.2f}%")
if (keras_acc - tflite_acc) < 0.02:
    print("✅ Quantization accuracy loss is acceptable (<2%)")
else:
    print("⚠️  Accuracy drop is significant — consider using float16 quantization instead")

# ================================================================
# MODEL SIZE COMPARISON
# ================================================================
sizes = {
    'Keras (.keras)':  os.path.getsize(keras_path),
    'H5 (.h5)':        os.path.getsize(h5_path),
    'TFLite float32':  os.path.getsize(f32_path),
    'TFLite INT8':     os.path.getsize(int8_path),
}

print(f"\n{'='*50}")
print(f"MODEL SIZE COMPARISON")
print(f"{'='*50}")
print(f"{'Format':<20} {'Size':>10}")
print(f"{'-'*32}")
for name, size in sizes.items():
    print(f"{name:<20} {size/1024:>8.1f} KB")
print(f"\nCompression ratio: {sizes['TFLite float32'] / sizes['TFLite INT8']:.1f}× (float32 → INT8)")
print(f"INT8 model: {sizes['TFLite INT8']} bytes — easily fits in RPi 5 L2 cache (512 KB per core)")

In [ ]:
# ================================================================
# SUMMARY OF SAVED FILES
# ================================================================
print("All files saved to:", OUTPUT_DIR)
print()
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    size = os.path.getsize(fpath)
    print(f"  {f:<40} {size/1024:>8.1f} KB")

print(f"\n{'='*50}")
print("FILES TO COPY TO RASPBERRY PI 5:")
print(f"{'='*50}")
print(f"  1. asl_mlp_int8.tflite    ← Primary model (INT8 quantized)")
print(f"  2. label_map.json          ← Class index → label name")
print(f"  3. scaler_mean.npy         ← Feature normalization mean")
print(f"  4. scaler_scale.npy        ← Feature normalization std")
print(f"\nOptional (fallback):")
print(f"  5. asl_mlp_float32.tflite  ← Float32 model (if INT8 has issues)")
print(f"  6. asl_mlp.h5              ← Full Keras model (for retraining)")

## Raspberry Pi 5 Deployment Guide

### Install dependencies (Debian Bookworm, Python 3.11):

```bash
pip install tflite-runtime numpy pandas scikit-learn
```

> **Note:** If `tflite-runtime` is unavailable for your platform, install the full TensorFlow package:
> `pip install tensorflow` (heavier, but guaranteed to work on ARM64).

### Required files (copy from Google Drive → RPi 5):

| File | Purpose |
|------|---------|
| `asl_mlp_int8.tflite` | INT8 quantized model (primary) |
| `label_map.json` | Class index → label name mapping |
| `scaler_mean.npy` | Feature normalization mean |
| `scaler_scale.npy` | Feature normalization standard deviation |

### Performance expectations on RPi 5:

- **Inference latency:** <1 ms per prediction (INT8 on Cortex-A76)
- **Model RAM:** <50 KB (fits entirely in L2 cache)
- **Total RAM overhead:** ~5–10 MB (interpreter + numpy)
- **Thermal:** No throttling risk — workload is trivially small

In [ ]:
# ================================================================
# SAVE RPi 5 INFERENCE SCRIPT — ready to run on Raspberry Pi 5
# ================================================================
# This creates a self-contained Python script that only needs:
#   pip install tflite-runtime numpy
# Place it in the same directory as the model files on RPi 5.
# ================================================================

rpi5_script = '''#!/usr/bin/env python3
"""
ASL Hand Sign Classifier — Raspberry Pi 5 Inference Script
============================================================
Requires: pip install tflite-runtime numpy
           (on Debian Bookworm / Python 3.11)

Model:   INT8 quantized MLP (~few KB)
Latency: <1 ms per inference on Cortex-A76
RAM:     ~5-10 MB total (interpreter + numpy)

Place this script alongside:
  - asl_mlp_int8.tflite
  - label_map.json
  - scaler_mean.npy
  - scaler_scale.npy
"""

import numpy as np
import json
import time

# --- Use tflite_runtime for minimal footprint on RPi 5 ---
# Falls back to full TensorFlow if tflite_runtime is not installed
try:
    from tflite_runtime.interpreter import Interpreter
except ImportError:
    from tensorflow.lite.python.interpreter import Interpreter

# ============================================================
# Load model and preprocessing parameters
# ============================================================
MODEL_PATH = "asl_mlp_int8.tflite"

interpreter = Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()
input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Load scaler params (saved as numpy arrays — no sklearn needed on RPi 5)
scaler_mean  = np.load("scaler_mean.npy")
scaler_scale = np.load("scaler_scale.npy")

# Load label mapping
with open("label_map.json") as f:
    label_map = json.load(f)


def predict_sign(landmarks_63):
    """
    Predict ASL sign from 63 MediaPipe hand landmark values.

    Args:
        landmarks_63: numpy array of shape (63,) —
                      flattened [x0, y0, z0, ..., x20, y20, z20]

    Returns:
        (predicted_label, confidence, top3_predictions)
    """
    # 1. Normalize using saved scaler parameters
    x = ((landmarks_63 - scaler_mean) / scaler_scale).astype(np.float32)
    x = x.reshape(1, 63)

    # 2. Run inference
    interpreter.set_tensor(input_details[0]["index"], x)
    interpreter.invoke()
    probs = interpreter.get_tensor(output_details[0]["index"])[0]

    # 3. Decode prediction
    top_idx    = int(np.argmax(probs))
    confidence = float(probs[top_idx])
    label      = label_map[str(top_idx)]

    # 4. Top-3 predictions (for "did you mean...?" UX)
    top3_idxs = np.argsort(probs)[-3:][::-1]
    top3 = [(label_map[str(int(i))], float(probs[i])) for i in top3_idxs]

    return label, confidence, top3


# ============================================================
# Quick benchmark — run on RPi 5 to check latency
# ============================================================
if __name__ == "__main__":
    dummy = np.random.randn(63).astype(np.float32)
    label, conf, top3 = predict_sign(dummy)
    print(f"Test prediction: {label} ({conf:.1%})")
    print(f"Top 3: {top3}")

    # Latency benchmark (1000 iterations)
    times = []
    for _ in range(1000):
        t0 = time.perf_counter()
        predict_sign(dummy)
        times.append(time.perf_counter() - t0)

    print(f"\\nLatency over 1000 calls:")
    print(f"  Mean: {np.mean(times)*1000:.2f} ms")
    print(f"  P50:  {np.percentile(times, 50)*1000:.2f} ms")
    print(f"  P99:  {np.percentile(times, 99)*1000:.2f} ms")
'''

rpi5_script_path = os.path.join(OUTPUT_DIR, "rpi5_inference.py")
with open(rpi5_script_path, 'w') as f:
    f.write(rpi5_script)
print(f"✅ RPi 5 inference script saved: {rpi5_script_path}")
print("   Copy to RPi 5 alongside model files and run: python3 rpi5_inference.py")